## This notebook requires GPU

LoRA fine-tuning requires a GPU. Set your Colab runtime to T4:

**Runtime > Change Runtime Type > T4 GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msds-marketing-analytics/colab-notebooks/blob/main/LLMs/MSDSTextClassification_FineTuningUnsloth.ipynb)

# Fine-Tuning LLMs with LoRA and Unsloth

Full fine-tuning of a large language model updates every parameter — billions
of weights, requiring enormous GPU memory and compute. **LoRA (Low-Rank
Adaptation)** offers a practical alternative: freeze the original model and
train only a small set of adapter weights. **Unsloth** makes this even faster
by fusing kernels and optimizing memory usage, enabling fine-tuning of 7B+
parameter models on a single T4 GPU.

This notebook demonstrates the complete workflow — from applying LoRA to a
model, through actual training, to evaluating the results and saving the adapter.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain what LoRA does** and why it is more efficient than full fine-tuning
2. **Apply LoRA adapters** to a pre-trained model using Unsloth and inspect what changed
3. **Prepare data** in instruction-tuning format for causal language model training
4. **Fine-tune a 7B model** with LoRA on a single T4 GPU and observe training loss decrease
5. **Evaluate before and after** training to measure the effect of fine-tuning
6. **Save and load LoRA adapters** separately from the base model

## Approach

We use Mistral 7B in 4-bit quantization via Unsloth. Despite having 7 billion
parameters, 4-bit quantization compresses the model to ~5GB, fitting comfortably
on a T4 GPU. LoRA then trains only ~0.5% of the parameters. We fine-tune on IMDB
sentiment data formatted as instructions, then measure whether the model learned
to classify sentiment.

In [ ]:
!pip install -q datasets evaluate scikit-learn matplotlib
!pip install unsloth

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import time
import os
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## Part 1: Loading the Base Model with Unsloth

We load Mistral 7B in 4-bit quantization through Unsloth. The model has 7 billion
parameters, but 4-bit quantization compresses each parameter from 16 bits to 4 bits,
reducing VRAM usage from ~14GB to ~5GB. Unsloth handles this automatically.

**A note on Unsloth's role here:** The 4-bit quantization comes from `bitsandbytes`,
the LoRA adapters come from `peft`, and the training loop comes from `transformers`.
Unsloth wraps these libraries and adds fused attention kernels and optimized gradient
checkpointing that speed up training ~2x and reduce memory usage ~40%. For our short
training run (100 steps), the wall-clock savings are modest — minutes rather than hours.
Unsloth's value scales with training duration and model size: a multi-hour fine-tuning
run on a 13B or 70B model is where the 2x speedup becomes essential.

In [ ]:
model_name = "unsloth/mistral-7b-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=256,
    dtype=None,       # auto-detect
    load_in_4bit=True,
)

# Mistral doesn't have a pad token — set it to EOS
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Parameters: {total_params:,}")
print(f"Quantization: 4-bit (via bitsandbytes)")
print(f"\nAll {total_params:,} parameters are currently trainable.")
print(f"Full fine-tuning would update every one of them — and require ~28GB of VRAM.")
print(f"LoRA will reduce this dramatically.")

---
## Part 2: Applying LoRA

### What LoRA Does

Instead of updating a full weight matrix **W** (e.g., 4096×4096 = 16.8M parameters),
LoRA learns two small matrices **A** (4096×16) and **B** (16×4096) whose product
approximates the weight update: **W' = W + BA**.

With rank `r=16`, those two matrices have only 4096×16 + 16×4096 = **131,072 parameters**
— less than 1% of the original matrix. The original weights stay frozen; only A and B
are trained.

### Key Parameters

- **r** (rank): Size of the low-rank matrices. Higher = more capacity, more parameters.
- **lora_alpha**: Scaling factor. Controls how much the adapter contributes.
- **target_modules**: Which layers get LoRA adapters. We target all attention
  projections (Q, K, V, O) and the feed-forward layers (gate, up, down).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Show what changed
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print("LoRA applied via Unsloth.")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Frozen parameters:    {frozen:,}")
print(f"  Trainable:            {100 * trainable / (trainable + frozen):.2f}%")
print(f"\nWe are training {trainable:,} parameters instead of {trainable + frozen:,}.")
print(f"That is a {(trainable + frozen) / trainable:.0f}x reduction in trainable parameters.")

In [ ]:
# Inspect which layers got LoRA adapters (show first 3 layers)
print("LoRA adapter layers (first 3 of 32):")
print("-" * 60)
count = 0
for name, module in model.named_modules():
    if "lora_A" in name and hasattr(module, 'weight'):
        print(f"  {name}: {tuple(module.weight.shape)}")
        count += 1
        if count >= 6:  # 2 per transformer layer (A matrix only), show 3 layers
            print(f"  ... ({sum(1 for n, _ in model.named_modules() if 'lora_A' in n and hasattr(_, 'weight'))} adapter matrices total)")
            break

---
## Part 3: Data Preparation

We format IMDB reviews as instruction-tuning examples. The model learns to
generate the correct label ("positive" or "negative") after seeing the
instruction and review text.

The format:
```
Classify the sentiment: positive or negative.
Review: <review text>
Sentiment: <label>
```

During training, the model sees the full sequence (including the label) and
learns to predict each token. At inference time, we give it everything except
the label and let it generate.

In [ ]:
# Load IMDB data
train_raw = load_dataset("imdb", split="train").shuffle(seed=42).select(range(1000))
test_raw = load_dataset("imdb", split="test").shuffle(seed=42).select(range(50))

print(f"Training examples: {len(train_raw)}")
print(f"Test examples: {len(test_raw)}")

# Format as instructions
def format_example(example):
    label = "positive" if example["label"] == 1 else "negative"
    return {
        "text": (
            "Classify the sentiment: positive or negative.\n"
            f"Review: {example['text'][:300]}\n"
            f"Sentiment: {label}"
        ),
        "label": example["label"]
    }

train_formatted = train_raw.map(format_example)
test_formatted = test_raw.map(format_example)

print(f"\nSample formatted example:")
print("-" * 40)
print(train_formatted[0]["text"][:300] + "...")
print("-" * 40)

In [ ]:
# Tokenize for training
def tokenize_fn(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    # For causal LM training, labels = input_ids
    # The model learns to predict each next token
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = train_formatted.map(
    tokenize_fn,
    remove_columns=train_formatted.column_names
)

print(f"Tokenized {len(tokenized_train)} training examples")
print(f"Sequence length: {len(tokenized_train[0]['input_ids'])} tokens")

---
## Part 4: Baseline — Before Training

Before fine-tuning, Mistral 7B is a general-purpose language model. It may
have some ability to follow instructions (since the base model saw instruction-like
text during pre-training), but it was not specifically trained on our sentiment
task. Let's see how it does.

In [ ]:
# Suppress a noisy deprecation warning from transformers' attention mask code
# (a formatting bug in their logger — not our code)
import logging
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

def evaluate_model(model, tokenizer, test_data, label=""):
    """Classify test examples and measure accuracy.

    Generates a response to the classification prompt and parses it
    for 'positive' or 'negative'. Outputs that contain neither are
    counted as unparseable.
    """
    FastLanguageModel.for_inference(model)
    predictions = []
    true_labels = []
    sample_outputs = []

    for example in test_data:
        prompt = (
            "Classify the sentiment: positive or negative.\n"
            f"Review: {example['text'][:300]}\n"
            "Sentiment:"
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=10, do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(
            outputs[0][len(inputs['input_ids'][0]):],
            skip_special_tokens=True
        ).strip().lower()

        if len(sample_outputs) < 5:
            sample_outputs.append(generated)

        if "positive" in generated:
            predictions.append(1)
        elif "negative" in generated:
            predictions.append(0)
        else:
            predictions.append(-1)
        true_labels.append(example["label"])

    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    parseable = predictions != -1

    acc = accuracy_score(true_labels[parseable], predictions[parseable]) if parseable.sum() > 0 else 0.0

    print(f"\n{label}")
    print(f"  Parseable: {parseable.sum()}/{len(predictions)}")
    print(f"  Accuracy (on parseable): {acc:.3f}")
    print(f"  Sample outputs: {sample_outputs[:5]}")

    return {"accuracy": acc, "parseable": int(parseable.sum()), "total": len(predictions)}

baseline = evaluate_model(model, tokenizer, test_raw, "BASELINE (before training)")

---
## Part 5: Fine-Tuning

Now we train the LoRA adapters. Only the adapter parameters (the small A and B
matrices) are updated; the original 7B model weights stay frozen.

We train for 100 steps with sequences capped at 256 tokens, which should
complete in about 5-10 minutes on a T4 GPU. For better accuracy, see the
commented-out configuration with 200 steps and 512 token sequences (expect
~25 minutes).

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./lora-sentiment",
    max_steps=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="no",
    report_to="none",
    optim="adamw_8bit",
)

# For better accuracy (~92%), uncomment the following and set
# max_seq_length=512 in Part 1 and max_length=512 in Part 3.
# Expect ~25 minutes of training time.
#
# training_args = TrainingArguments(
#     output_dir="./lora-sentiment",
#     max_steps=200,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     warmup_steps=10,
#     logging_steps=20,
#     fp16=not torch.cuda.is_bf16_supported(),
#     bf16=torch.cuda.is_bf16_supported(),
#     save_strategy="no",
#     report_to="none",
#     optim="adamw_8bit",
# )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print(f"Training config:")
print(f"  Steps: {training_args.max_steps}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: 8-bit AdamW (memory efficient)")
print()

start = time.time()
train_result = trainer.train()
train_time = time.time() - start

# Get the last logged loss (not the average, which is what train_result.training_loss reports)
last_loss = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]

print(f"\nTraining complete in {train_time:.0f}s")
print(f"Final loss (last step): {last_loss:.4f}")

In [ ]:
# Plot training loss
log_history = trainer.state.log_history
train_logs = [h for h in log_history if "loss" in h]

if train_logs:
    steps = [h["step"] for h in train_logs]
    losses = [h["loss"] for h in train_logs]

    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, "o-", color="steelblue")
    plt.xlabel("Step")
    plt.ylabel("Training Loss")
    plt.title("LoRA Fine-Tuning: Training Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Loss decreased from {losses[0]:.4f} to {losses[-1]:.4f}")
else:
    print("No training logs available.")

---
## Part 6: Evaluation — After Training

Let's run the same classification task on the same test examples and see
whether the model improved.

In [ ]:
after = evaluate_model(model, tokenizer, test_raw, "AFTER TRAINING")

print("\n" + "=" * 50)
print("Before vs. After Fine-Tuning")
print("=" * 50)
print(f"  {'':20} {'Before':>10} {'After':>10}")
print(f"  {'Parseable outputs':20} {baseline['parseable']:>8}/{baseline['total']} {after['parseable']:>8}/{after['total']}")
print(f"  {'Accuracy':20} {baseline['accuracy']:>10.3f} {after['accuracy']:>10.3f}")

if after['parseable'] > baseline['parseable']:
    print(f"\nThe model learned to output 'positive'/'negative' — it went from")
    print(f"{baseline['parseable']} to {after['parseable']} parseable outputs out of {after['total']}.")
if after['accuracy'] > baseline['accuracy']:
    print(f"Classification accuracy improved from {baseline['accuracy']:.3f} to {after['accuracy']:.3f}.")

---
## Part 7: Saving and Loading LoRA Adapters

One of LoRA's key advantages: you save only the adapter weights, not the
full model. The adapter is tiny — often less than 1% of the base model's
size. To use it later, you load the base model and apply the adapter on top.

In [ ]:
# Save just the LoRA adapter
adapter_dir = "./lora-sentiment-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Measure file sizes
adapter_files = [f for f in os.listdir(adapter_dir)
                 if f.endswith(('.safetensors', '.bin'))]
adapter_size = sum(os.path.getsize(os.path.join(adapter_dir, f))
                   for f in adapter_files)
full_model_size = total_params * 4  # FP32 bytes

print(f"Saved to: {adapter_dir}/")
print(f"  Adapter files: {adapter_files}")
print(f"  Adapter size: {adapter_size / 1024:.1f} KB")
print(f"  Full model would be: ~{full_model_size / 1024**2:.0f} MB")
print(f"  Adapter is {100 * adapter_size / full_model_size:.2f}% of the full model")

In [ ]:
# Demonstrate loading: start from a fresh base model + apply adapter
from peft import PeftModel

print("Loading fresh base model via Unsloth...")
fresh_model, fresh_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=256,
    dtype=None,
    load_in_4bit=True,
)
fresh_model.config.pad_token_id = tokenizer.eos_token_id

print("Applying saved LoRA adapter...")
loaded_model = PeftModel.from_pretrained(fresh_model, adapter_dir)

# Verify it works the same as our trained model
loaded_result = evaluate_model(loaded_model, tokenizer, test_raw, "LOADED ADAPTER")

print(f"\nAccuracy matches trained model: {loaded_result['accuracy'] == after['accuracy']}")

---
## Key Takeaways

1. **LoRA trains a tiny fraction of parameters.** Instead of updating all 7B
   parameters, we trained only the adapter matrices — a massive reduction in
   trainable parameters. This is what makes fine-tuning a 7B model possible
   on a single T4 GPU.

2. **The base model's weights never change.** Only the adapter matrices A and B
   are updated. This means you can share one base model across many tasks,
   each with its own small adapter.

3. **4-bit quantization + LoRA = QLoRA.** The base model is stored in 4-bit
   precision (~5GB), and LoRA trains small FP16 adapter weights on top. This
   combination makes 7B+ models accessible on consumer hardware.

4. **Unsloth accelerates training.** Fused kernels, optimized gradient
   checkpointing, and 8-bit optimizers reduce both memory usage and training
   time compared to standard PEFT.

5. **Adapters are tiny and portable.** The LoRA adapter is a few MB compared
   to the multi-GB base model — easy to store, version, and share.

6. **Fine-tuning teaches the model a task format.** The base model had general
   language ability; after LoRA training, it learned to output sentiment labels
   in response to our specific prompt format.

## Exercises

1. **Vary the rank**: Try `r=4`, `r=8`, and `r=32`. How does the rank affect
   the number of trainable parameters, training speed, and classification accuracy?

2. **More training**: Increase `max_steps` to 500 or 1000. Does accuracy keep
   improving, or does it plateau?

3. **Different target modules**: Try targeting only attention layers (`q_proj`,
   `v_proj`) vs. all layers. Does targeting more layers help?

4. **More training data**: Increase from 1000 to 5000 training examples.
   How much does additional data improve results?

5. **Merge the adapter**: Use `model.merge_and_unload()` to fold the LoRA
   weights into the base model. Compare the merged model's file size with
   the adapter-based approach.